# M1 · Load OLMoE and look at the router

**Goal:** load the MoE model and see, for every token, which experts the router picks.

**Kaggle:** GPU T4 x1, Internet On. Takes ~10 minutes (most of it is downloading 14 GB).

### Words we use
- **Expert**: a small feed-forward network. OLMoE has **64 experts in each of its 16 layers**.
- **Router**: a tiny layer that scores the 64 experts for each token and picks the **top 8**.
- So each token uses only 8/64 = 12.5% of the experts → that is why MoE is cheap to *compute*.
  But all 64 experts must still be *stored* somewhere → that is the memory problem we study.

In [ ]:
# ---- Setup: get the latest project code (run this cell first, every session) ----
import os, sys

REPO_URL = "https://github.com/YOUR-TEAM/moe-offload.git"   # <-- change to your team's repo
# Private repo? Add a Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

if os.path.exists("/kaggle"):                      # running on Kaggle
    import glob, shutil
    CODE_DIR = "/kaggle/working/moe-offload"
    # Option A: the code was uploaded as a Kaggle dataset ("moe-offload-code") and added as an input
    uploaded = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/moe-offload-code/config.py", recursive=True)]
    if uploaded:
        shutil.copytree(uploaded[0], CODE_DIR, dirs_exist_ok=True)
        print("using code from dataset:", uploaded[0])
    # Option B: get the code from GitHub
    elif not os.path.exists(CODE_DIR):
        !git clone -q {REPO_URL} {CODE_DIR}         # first time: download the code
    else:
        !git -C {CODE_DIR} pull -q                  # later: get the newest commits
    os.chdir(CODE_DIR)
    !pip install -q -r requirements.txt
else:                                              # running on your own computer
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                             # go to the repo root

sys.path.insert(0, os.getcwd())                    # so "import config" and "from src..." work
!git log -1 --format="Code version: %h  %s"

## Load the model
We use **float16** because the T4 GPU does not support bfloat16.

In [ ]:
import torch
import config
from src.traces import load_moe, route, chunk_loss

model, tok = load_moe()          # downloads the model the first time
cfg = model.config
print("layers:", cfg.num_hidden_layers, "| experts per layer:", cfg.num_experts,
      "| experts per token:", cfg.num_experts_per_tok)
print("GPU memory used by the weights: %.1f GB" % (torch.cuda.memory_allocated() / 1e9))

## How big is one expert?
Each expert has 3 weight matrices (gate, up, down) of size 2048 × 1024.
In transformers 5.x all 64 experts of a layer are stored together in two big tensors.

In [ ]:
experts = model.model.layers[0].mlp.experts
print("gate_up_proj:", tuple(experts.gate_up_proj.shape), " (64 experts x [2*1024 x 2048])")
print("down_proj   :", tuple(experts.down_proj.shape), " (64 experts x [2048 x 1024])")

params_per_expert = experts.gate_up_proj[0].numel() + experts.down_proj[0].numel()
mb = params_per_expert * 2 / 1e6               # 2 bytes per number in float16
print(f"one expert = {params_per_expert/1e6:.1f} M numbers = {mb:.1f} MB")
print(f"all experts = 16 layers x 64 x {mb:.1f} MB = {16*64*mb/1e3:.1f} GB")

## Which experts does each token use?
`route()` runs the model once and returns the chosen experts:
shape **[batch, tokens, layers, 8]**.

In [ ]:
text = "The cat sat on the mat. def add(a, b): return a + b"
ids = tok(text, return_tensors="pt").input_ids.cuda()
experts_used, _ = route(model, ids)
experts_used = experts_used[0].cpu()           # drop the batch dimension -> [tokens, layers, 8]
print("shape:", tuple(experts_used.shape))

# print the experts of layer 0 and layer 15 for every token
for i, token_id in enumerate(ids[0]):
    word = repr(tok.decode(token_id))
    print(f"{word:12s} layer 0: {experts_used[i, 0].tolist()}   layer 15: {experts_used[i, 15].tolist()}")

**Look for:** do neighbouring tokens share experts? Do code tokens use different
experts from English tokens? If tokens re-use experts, a cache will help.

## Sanity check: perplexity
Perplexity measures how well the model predicts the next token (lower = better).
For OLMoE on WikiText we expect roughly **10–20**. A huge number means something is broken.

In [ ]:
from src.data import make_chunks
import numpy as np

chunks = make_chunks(tok, "wikitext")[:20]     # 20 chunks of 512 tokens
loss_sum, n_tokens = 0.0, 0
for chunk in chunks:
    ids = chunk.view(1, -1).cuda()
    _, logits = route(model, ids)
    l, n = chunk_loss(logits, ids)
    loss_sum, n_tokens = loss_sum + l, n_tokens + n
print("WikiText perplexity: %.2f" % np.exp(loss_sum / n_tokens))

## ✅ Done when
- you can print the 8 experts per token per layer, and
- the perplexity is in a sensible range.

Write 3–4 sentences in `docs/M1.md` about what you saw.